<a href="https://colab.research.google.com/github/toddbalwinski/ds2002-fa26/blob/main/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [15]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [16]:
# TODO

df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)

print("The 400 orders generated $8520 in total revenue from 783 total units sold.")

Total revenue: 8520.0
Total units: 783
The 400 orders generated $8520 in total revenue from 783 total units sold.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [23]:
# TODO
category_summary = (
    df.groupby('category', as_index=False)
      .agg(revenue=('revenue', 'sum'))
)

category_summary['share_of_total'] = (
    category_summary['revenue'] / total_revenue * 100
).round(1)

category_summary = category_summary.sort_values(
    'revenue',
    ascending=False
)

print(category_summary)

print("Food was the highest grossing category with ~50% of the revenue and RainGear was the least with ~10%")


   category  revenue  share_of_total
1      Food   4293.0            50.4
2     Merch   1771.5            20.8
0     Drink   1554.0            18.2
3  RainGear    901.5            10.6
Food was the highest grossing category with ~50% of the revenue and RainGear was the least with ~10%


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [26]:
# Q3
vendor_summary = (
    df.groupby('vendor_id', as_index=False)
      .agg(
          avg_order_revenue=('revenue', 'mean'),
          order_count=('revenue', 'count')
      )
)

vendor_summary['avg_order_revenue'] = (
    vendor_summary['avg_order_revenue'].round(2)
)

vendor_summary = vendor_summary.sort_values(
    'avg_order_revenue',
    ascending=False
)

print(vendor_summary)

print("V-01 has the highest average order revenue at 22.6 with an order count of 94 which is right in line with the other vendors order counts")

  vendor_id  avg_order_revenue  order_count
0      V-01              22.60           94
3      V-18              21.75          108
1      V-05              20.58           93
2      V-10              20.31          105
V-01 has the highest average order revenue at 22.6 with an order count of 94 which is right in line with the other vendors order counts


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [30]:
merch_revenue = category_summary.loc[
    category_summary['category'] == 'Merch',
    'revenue'
].iloc[0]

merch_share = (merch_revenue / total_revenue * 100).round(1)

print(merch_share)

print("Merch accounts for 20.8% of total revenue.")

20.8
Merch accounts for 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [32]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

original_rows = len(df)
original_revenue = df['revenue'].sum()

df_named = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Original rows:", original_rows)
print("Rows after merge:", len(df_named))

print("Original revenue:", original_revenue)
print("Revenue after merge:", df_named['revenue'].sum())

unmatched = df_named.loc[
    df_named['vendor_name'].isna(),
    'vendor_id'
].unique()

print("Unmatched vendor IDs:", unmatched)

df_named['vendor_name'] = df_named['vendor_name'].fillna('Unknown Vendor')

Original rows: 400
Rows after merge: 400
Original revenue: 8520.0
Revenue after merge: 8520.0
Unmatched vendor IDs: ['V-18']


**The unmatched vendor is V-18, and I decided to fill all the unknown rows with "unknown vendor"**

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [33]:
vendor_pivot = pd.pivot_table(
    df_named,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

vendor_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [22]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

I think that that the vendors should improve their sales strategy for merchandise as it only made up 20% of the profits and food more than doubled it. Maybe opening up more vendors or releasing new trening products would help this. I also think that they need to improve the sales of their drinks as that again at less than half of foods 50% with only an 18% share of revenue. This could be done by having people with drinks walk into the crowds to sell drinks to people in their seats to push sales

I think that our vendor lookup table was the least trustworthy as we have a whole unknown vendor that is not accounted for, therefore calling into question who is selling what and whos recording/reporting it.